# Vital Sign Risk Model (Edge-Friendly) — End-to-End Training Notebook

This notebook trains a **tiny, deployable** model that predicts **Risk Category (High/Low)** from vital signs.

**Why this approach?**
- Uses a **Logistic Regression** classifier → small, fast, and easy to deploy on ESP32/Arduino as a few numbers (weights + bias).
- Includes **data checks**, **EDA**, **train/test split**, **scaling**, **training**, **evaluation**, and **export** steps.

---

## Dataset expected
A CSV file named:

- `human_vital_signs_dataset_2024.csv`

It should contain (at minimum):
- `Heart Rate`
- `Body Temperature`
- `Oxygen Saturation`
- `Risk Category` (values: `High Risk` / `Low Risk`)


## Stage 0 — Setup & Imports

In [ ]:
# Core data + ML imports
import os
import json
import math
import numpy as np
import pandas as pd

# Plotting (matplotlib only)
import matplotlib.pyplot as plt

# Scikit-learn utilities
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    average_precision_score,
)

# Optional: for saving artifacts locally
import joblib

# Make results reproducible
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Ready ✅")


## Stage 1 — Load the dataset

This cell:
1. Looks for the CSV in the current folder
2. Loads it into a pandas DataFrame
3. Shows basic info (rows, columns, sample records)


In [ ]:
DATA_PATH = "human_vital_signs_dataset_2024.csv"

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Could not find '{DATA_PATH}'.\n"
        "Put the CSV in the same folder as this notebook, or change DATA_PATH to the correct path."
    )

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("Columns:", list(df.columns))
display(df.head(5))


## Stage 2 — Validate schema (columns + label values)

We validate that:
- Required feature columns exist
- Label column exists
- Label contains the expected values (High Risk / Low Risk)

This prevents silent mistakes later in training.


In [ ]:
# --- Required inputs for the model ---
FEATURE_COLS = ["Heart Rate", "Body Temperature", "Oxygen Saturation"]
LABEL_COL = "Risk Category"

def validate_schema(dataframe: pd.DataFrame, feature_cols, label_col) -> None:
    """Raise a clear error if required columns are missing or labels are unexpected."""
    missing = [c for c in (feature_cols + [label_col]) if c not in dataframe.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    # Ensure label values are only what we expect
    allowed = {"High Risk", "Low Risk"}
    actual = set(dataframe[label_col].dropna().unique())
    unexpected = actual - allowed
    if unexpected:
        raise ValueError(
            f"Unexpected label values found: {sorted(unexpected)}\n"
            f"Allowed values are: {sorted(allowed)}"
        )

validate_schema(df, FEATURE_COLS, LABEL_COL)
print("Schema looks good ✅")


## Stage 3 — Data quality checks

We check:
- Missing values in model columns
- Basic stats (min/max/mean)
- Class balance (High vs Low)

Class balance matters because it impacts accuracy and thresholding.


In [ ]:
# Check missing values in the columns we care about
subset = FEATURE_COLS + [LABEL_COL]
missing_counts = df[subset].isna().sum()
print("Missing values per column:")
display(missing_counts)

# Basic stats for feature columns
print("\nFeature summary statistics:")
display(df[FEATURE_COLS].describe())

# Class balance
class_counts = df[LABEL_COL].value_counts()
print("\nClass counts:")
display(class_counts)

# Plot class balance (simple bar chart)
plt.figure()
class_counts.plot(kind="bar")
plt.title("Class Balance (Risk Category)")
plt.xlabel("Class")
plt.ylabel("Count")
plt.show()


## Stage 4 — Quick EDA (feature distributions)

We visualize distributions of the core vitals:
- Heart Rate
- Temperature
- SpO₂

This helps sanity-check ranges and detect weird outliers.


In [ ]:
def plot_histograms(dataframe: pd.DataFrame, cols, bins=30) -> None:
    """Plot one histogram per feature column."""
    for col in cols:
        plt.figure()
        dataframe[col].plot(kind="hist", bins=bins)
        plt.title(f"Distribution: {col}")
        plt.xlabel(col)
        plt.ylabel("Frequency")
        plt.show()

plot_histograms(df, FEATURE_COLS, bins=40)


## Stage 5 — Build X (features) and y (label)

We convert labels to numeric:
- High Risk → 1
- Low Risk  → 0


In [ ]:
# Features matrix (X)
X = df[FEATURE_COLS].copy()

# Numeric label (y)
y = (df[LABEL_COL] == "High Risk").astype(int)

print("X shape:", X.shape)
print("y distribution (0=Low, 1=High):")
display(y.value_counts())


## Stage 6 — Train/Test split

We use a stratified split so both sets keep similar class balance.

- Train: 80%
- Test:  20%


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,  # keeps class ratio stable across splits
)

print("Train size:", X_train.shape[0])
print("Test size :", X_test.shape[0])


## Stage 7 — Scale features (Standardization)

Logistic regression trains better when features are on similar scales.

We compute:
- mean and standard deviation from the training set only
- apply the same transform to the test set


In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)  # fit on train
X_test_scaled = scaler.transform(X_test)        # transform test using train stats

print("Scaler means:", scaler.mean_)
print("Scaler stds :", scaler.scale_)


## Stage 8 — Train the model (Logistic Regression)

This is a very small model:
- It learns one weight per feature + a bias term
- Great for embedded deployment

Tip:
If classes are imbalanced, `class_weight="balanced"` can help.
We’ll start without it, and you can switch it on if needed.


In [ ]:
model = LogisticRegression(
    solver="lbfgs",
    max_iter=2000,
    random_state=RANDOM_STATE,
    # class_weight="balanced",  # uncomment if you see class imbalance issues
)

model.fit(X_train_scaled, y_train)

print("Training complete ✅")
print("Weights:", model.coef_[0])
print("Bias   :", model.intercept_[0])


## Stage 9 — Evaluate (classification report + confusion matrix)

We evaluate on the **test set** (data the model never saw during training).


In [ ]:
# Predicted class (0/1) using default threshold 0.5
y_pred = model.predict(X_test_scaled)

print(classification_report(y_test, y_pred, target_names=["Low Risk", "High Risk"]))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Low Risk", "High Risk"])
disp.plot()
plt.title("Confusion Matrix (Threshold = 0.5)")
plt.show()


## Stage 10 — Evaluate probabilities (ROC + PR curves)

For health monitoring, you often care about **sensitivity/recall** (catch high risk) vs **false alarms**.

We’ll compute:
- ROC AUC
- PR AUC (Average Precision)
and plot ROC + Precision-Recall curves.


In [ ]:
# Probabilities for the positive class (High Risk = 1)
y_proba = model.predict_proba(X_test_scaled)[:, 1]

roc_auc = roc_auc_score(y_test, y_proba)
ap = average_precision_score(y_test, y_proba)

print(f"ROC AUC: {roc_auc:.4f}")
print(f"PR AUC (Average Precision): {ap:.4f}")

# ROC curve
fpr, tpr, roc_thresholds = roc_curve(y_test, y_proba)
plt.figure()
plt.plot(fpr, tpr)
plt.plot([0, 1], [0, 1], linestyle="--")
plt.title("ROC Curve")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.show()

# Precision-Recall curve
precision, recall, pr_thresholds = precision_recall_curve(y_test, y_proba)
plt.figure()
plt.plot(recall, precision)
plt.title("Precision-Recall Curve")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.show()


## Stage 11 — Choose a threshold (optional but useful)

Default threshold is **0.5**, but for rural clinics you may prefer:
- Higher recall (catch more high-risk cases)
- Accept more false alarms (depending on staffing/workload)

This cell finds the threshold that gives a desired recall target.


In [ ]:
TARGET_RECALL = 0.90  # change to 0.95 if you want even higher sensitivity

precision, recall, thresholds = precision_recall_curve(y_test, y_proba)

# thresholds array is length-1 compared to precision/recall
# We'll choose the first threshold where recall >= target
chosen_threshold = None
for p, r, t in zip(precision[:-1], recall[:-1], thresholds):
    if r >= TARGET_RECALL:
        chosen_threshold = float(t)
        chosen_precision = float(p)
        chosen_recall = float(r)
        break

if chosen_threshold is None:
    print("Could not find a threshold that achieves the target recall. Try lowering TARGET_RECALL.")
else:
    print(f"Chosen threshold: {chosen_threshold:.4f}")
    print(f"At this threshold -> Precision: {chosen_precision:.4f}, Recall: {chosen_recall:.4f}")

    # Apply chosen threshold
    y_pred_custom = (y_proba >= chosen_threshold).astype(int)
    print("\nClassification report (custom threshold):")
    print(classification_report(y_test, y_pred_custom, target_names=["Low Risk", "High Risk"]))

    cm2 = confusion_matrix(y_test, y_pred_custom, labels=[0, 1])
    disp2 = ConfusionMatrixDisplay(confusion_matrix=cm2, display_labels=["Low Risk", "High Risk"])
    disp2.plot()
    plt.title(f"Confusion Matrix (Threshold = {chosen_threshold:.3f})")
    plt.show()


## Stage 12 — Save artifacts (model + scaler)

We save:
- The trained model
- The scaler (means/stds)

These are needed if you want to load and use the model later on a PC/Raspberry Pi.
For microcontrollers, we’ll also export raw numbers next.


In [ ]:
ARTIFACT_DIR = "artifacts"
os.makedirs(ARTIFACT_DIR, exist_ok=True)

model_path = os.path.join(ARTIFACT_DIR, "logreg_model.joblib")
scaler_path = os.path.join(ARTIFACT_DIR, "scaler.joblib")

joblib.dump(model, model_path)
joblib.dump(scaler, scaler_path)

print("Saved:")
print(" -", model_path)
print(" -", scaler_path)


## Stage 13 — Export for ESP32/Arduino (weights + bias + scaling params)

On a microcontroller, you usually **don’t run scikit-learn**.
Instead you export the few numbers and implement:

1. Standardize inputs: `(x - mean) / std`
2. Score: `score = w·x + b`
3. Sigmoid: `p = 1 / (1 + exp(-score))`
4. Compare to threshold

This cell exports:
- JSON file (easy to parse)
- C header snippet (copy/paste into Arduino code)


In [ ]:
# Decide which threshold to export:
# - Use the custom threshold if it exists, otherwise default to 0.5
export_threshold = 0.5
if "chosen_threshold" in globals() and chosen_threshold is not None:
    export_threshold = float(chosen_threshold)

export_data = {
    "feature_cols": FEATURE_COLS,
    "scaler_mean": scaler.mean_.tolist(),
    "scaler_std": scaler.scale_.tolist(),
    "weights": model.coef_[0].tolist(),
    "bias": float(model.intercept_[0]),
    "threshold": export_threshold,
}

json_path = os.path.join(ARTIFACT_DIR, "edge_params.json")
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(export_data, f, indent=2)

print("Exported JSON params ->", json_path)
display(export_data)

# Generate a C header snippet
def to_c_array(name: str, values) -> str:
    values_str = ", ".join([f"{float(v):.10f}" for v in values])
    return f"static const float {name}[{len(values)}] = {{{values_str}}};"

c_lines = []
c_lines.append("// Auto-generated edge params for logistic regression risk model")
c_lines.append("#pragma once")
c_lines.append("")
c_lines.append(to_c_array("SCALER_MEAN", export_data["scaler_mean"]))
c_lines.append(to_c_array("SCALER_STD", export_data["scaler_std"]))
c_lines.append(to_c_array("WEIGHTS", export_data["weights"]))
c_lines.append(f"static const float BIAS = {export_data['bias']:.10f}f;")
c_lines.append(f"static const float THRESHOLD = {export_data['threshold']:.10f}f;")
c_lines.append("")
c_lines.append("// Feature order:")
for i, col in enumerate(FEATURE_COLS):
    c_lines.append(f"//   {i}: {col}")

header_text = "\n".join(c_lines)

header_path = os.path.join(ARTIFACT_DIR, "edge_params.h")
with open(header_path, "w", encoding="utf-8") as f:
    f.write(header_text)

print("Exported C header ->", header_path)
print("\n--- edge_params.h preview ---\n")
print(header_text[:1200])  # preview first part


## Stage 14 — (Optional) Minimal inference function (Python)

This mirrors what you'll do on ESP32/Arduino:
- Standardize inputs
- Compute score
- Sigmoid → probability
- Threshold → risk class


In [ ]:
def sigmoid(x: float) -> float:
    """Numerically-stable sigmoid for edge-like inference."""
    # Clamp input to avoid overflow in exp() for very large magnitude values
    x = float(np.clip(x, -50, 50))
    return 1.0 / (1.0 + math.exp(-x))

def predict_edge_like(hr: float, temp: float, spo2: float, params: dict) -> dict:
    """Return probability + class using exported params, like an embedded device would."""
    x = np.array([hr, temp, spo2], dtype=float)

    mean = np.array(params["scaler_mean"], dtype=float)
    std = np.array(params["scaler_std"], dtype=float)
    w = np.array(params["weights"], dtype=float)
    b = float(params["bias"])
    thresh = float(params["threshold"])

    x_scaled = (x - mean) / std
    score = float(np.dot(w, x_scaled) + b)
    prob = sigmoid(score)
    pred = 1 if prob >= thresh else 0

    return {
        "prob_high_risk": prob,
        "predicted_class": "High Risk" if pred == 1 else "Low Risk",
        "threshold": thresh,
        "feature_order": params["feature_cols"],
    }

# Quick test using a sample row from the dataset
sample = df.sample(1, random_state=RANDOM_STATE).iloc[0]
result = predict_edge_like(
    hr=float(sample["Heart Rate"]),
    temp=float(sample["Body Temperature"]),
    spo2=float(sample["Oxygen Saturation"]),
    params=export_data
)

print("Sample input:")
print(sample[FEATURE_COLS].to_dict())
print("\nEdge-like inference result:")
display(result)


## What you can do next (project-ready ideas)

1. **Add more sensors/features** (respiratory rate, blood pressure) and compare performance.
2. Build a **real-time windowing** dataset (e.g., last 10 seconds) if your hardware streams data.
3. If you want *true anomaly detection*, train on **Low Risk** only and detect deviations — but you’ll need more abnormal examples or time-series data.

---

If you want, paste your intended ESP32/Arduino setup (what sensors + sampling rate), and I’ll generate the embedded inference code too.
